### RAG Pipeline - Data Ingestion to Vector DB pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
# Read all PDF files in the directory
def process_all_pdfs(pdfs_directory):
    """Process all PDF files in a directory."""
    all_documents = []
    pdf_dir = Path(pdfs_directory)
    
    # Find all pdf files recursively
    pdf_files = list(pdf_dir.glob('**/*.pdf'))
    
    print(f"Found {len(pdf_files)} PDF files to process.")
    
    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
            print(f"✓ Loaded {len(documents)} pages.")
            
        except Exception as e:
            print(f"✗ Error: {e}")
            
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all pdfs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 10 PDF files to process.
Processing: ..\data\pdfs\1. A_Review_of_Effectiveness_and_Efficiency_Methodology_of_Decision_Support_System_for_Selecting_Suppliers.pdf
✓ Loaded 7 pages.
Processing: ..\data\pdfs\10. Modeling_Dynamic_User_Preference_via_Dictionary_Learning_for_Sequential_Recommendation.pdf
✓ Loaded 13 pages.
Processing: ..\data\pdfs\2. A_Novel_Approach_for_Customer_Segmentation_and_Product_Recommendation_to_Boost_Sales_using_Machine_Learning.pdf
✓ Loaded 7 pages.
Processing: ..\data\pdfs\3. E-commerce_platform_based_on_Machine_Learning_Recommendation_System.pdf
✓ Loaded 4 pages.
Processing: ..\data\pdfs\4. Design_and_Implementation_of_a_Product_Recommendation_System_with_AssociationClustering_algorithms.pdf
✓ Loaded 9 pages.
Processing: ..\data\pdfs\5. An_Analysis_on_various_Machine_Learning_Algorithms_AI_amp_Nature_Inspired_Algorithms_for_modern_Inventory_Management.pdf
✓ Loaded 8 pages.
Processing: ..\data\pdfs\6. A_Novel_Time-Aware_Food_Recommender-System_Based_on_Deep

In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Neevia Document Converter Pro v7.0.0.82 (http://neevia.com); modified using iTextSharp 5.0.2 (c) 1T3XT BVBA; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'Certified by IEEE PDFeXpress at November 4, 2023 04:28:03', 'creationdate': '2023-11-04T04:28:00+00:00', 'meeting starting date': '10 Oct. 2023', 'moddate': '2023-12-14T17:33:51-05:00', 'ieee article id': '10346850', 'ieee issue id': '10346224', 'subject': '2023 International Conference on Computer Science and Emerging Technologies (CSET);2023; ; ;10.1109/CSET58993.2023.10346850', 'ieee publication id': '10346235', 'title': 'A Review of Effectiveness and Efficiency Methodology of Decision Support System for Selecting Suppliers', 'meeting ending date': '12 Oct. 2023', 'source': '..\\data\\pdfs\\1. A_Review_of_Effectiveness_and_Efficiency_Methodology_of_Decision_Support_System_for_Selecting_Suppliers.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_

In [8]:
# Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}\n...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs
    
chunks = split_documents(all_pdf_documents)
chunks

Split 106 documents into 729 chunks.

Example chunk:
Content: 979-8-3503-4173-7/23/$31.00 ©2023 IEEE 
2023 International Conference on Computer Science and Emerging Technologies (CSET) 
 
A Review of Effectiveness and Efficiency 
Methodology of Decision Support 
...
Metadata: {'producer': 'Neevia Document Converter Pro v7.0.0.82 (http://neevia.com); modified using iTextSharp 5.0.2 (c) 1T3XT BVBA; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'Certified by IEEE PDFeXpress at November 4, 2023 04:28:03', 'creationdate': '2023-11-04T04:28:00+00:00', 'meeting starting date': '10 Oct. 2023', 'moddate': '2023-12-14T17:33:51-05:00', 'ieee article id': '10346850', 'ieee issue id': '10346224', 'subject': '2023 International Conference on Computer Science and Emerging Technologies (CSET);2023; ; ;10.1109/CSET58993.2023.10346850', 'ieee publication id': '10346235', 'title': 'A Review of Effectiveness and Efficiency Methodology of Decision Support System for

[Document(metadata={'producer': 'Neevia Document Converter Pro v7.0.0.82 (http://neevia.com); modified using iTextSharp 5.0.2 (c) 1T3XT BVBA; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'Certified by IEEE PDFeXpress at November 4, 2023 04:28:03', 'creationdate': '2023-11-04T04:28:00+00:00', 'meeting starting date': '10 Oct. 2023', 'moddate': '2023-12-14T17:33:51-05:00', 'ieee article id': '10346850', 'ieee issue id': '10346224', 'subject': '2023 International Conference on Computer Science and Emerging Technologies (CSET);2023; ; ;10.1109/CSET58993.2023.10346850', 'ieee publication id': '10346235', 'title': 'A Review of Effectiveness and Efficiency Methodology of Decision Support System for Selecting Suppliers', 'meeting ending date': '12 Oct. 2023', 'source': '..\\data\\pdfs\\1. A_Review_of_Effectiveness_and_Efficiency_Methodology_of_Decision_Support_System_for_Selecting_Suppliers.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_

#### Embedding and VectorStore DB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\My GitHub\rachit404\YtRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers."""
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initialize the embedding manager
        
        Args:
            model_name: Hugging Face model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):        
        """Load the sentence transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.
        
        Args:
            texts: List of text strings to embed
        
        Returns:
            Numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

# Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


#### Vector Store

In [18]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store."""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store.
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB Client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or Create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
                )
            print(f"Vector store intialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise e
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.
        
        Args:
            documents: List of LangChain Document objects
            embeddings: corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings.")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)            
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
            
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection now: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
# Initialize vector store
vector_store = VectorStore()
vector_store                   

Vector store intialized. Collection: pdf_documents
Existing documents in collection: 0


In [21]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 729 texts...


Batches: 100%|██████████| 23/23 [00:43<00:00,  1.91s/it]


Generated embeddings with shape: (729, 384)
Adding 729 documents to vector store...
Successfully added 729 documents to vector store.
Total documents in collection now: 729


### Retriever Pipeline from VectorStore

In [24]:
class RAGRetriver:
    """Handles query based retrieval from the vector store."""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever.
        
        Args:
            vector_store: VectorStore containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.
        
        Args:
            query: User search query string
            top_k: Number of top results to return
            score_threshold: Minimum similarity score to threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1,                            
                        })
                        
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering).")
            else:
                print("No documents retrieved.")
                
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
        
# Initialize retriever
rag_retriever = RAGRetriver(vector_store, embedding_manager)
rag_retriever 

In [30]:
rag_retriever.retrieve("What is recommendation system?")

Retrieving documents for query: 'What is recommendation system?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.55it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering).


[{'id': 'doc_e33e1df1_205',
  'content': 'recommendation algorithms. The basic purpose of a \nrecommendation system is to boost product sales by providing \na relevant item to the customer and so improving total profit, \nwhich encompasses the functional goals of recommendation \nsystems including relevancy, serendipity and diversity [6]. A \nrecommendation system can help clients quickly find a wide \nrange of things that they are interested in. The popularity of \nthis effective suggestion system is growing by the day because \nit is simple and reliable for a client to purchase online and find \nthe best selections for them. \nThe Recommender system aims to provide users with \nproduct, service and information recommendations based on \ntheir interests, taking into account their needs and preferences. \nA user would undoubtedly prefer a website that suggests \nsomething beneficial to him over one that forces him to \nbrowse through the site to find the products they require. A \nreco